# EDA Part 2 - CA_1 Focused Analysis

**Date:** 2026-05-04  
**Scope:** CA_1 store × all products (3,049 series, 3 categories, 7 departments, 1,941 days)  
**Goal:** Detailed intermittent demand profiling, price dynamics, event effects, and train/val/test split visualization.  
**Feeds into:** LightGBM Tweedie objective justification, feature engineering decisions.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from retail_forecast.config import RAW_DATA_DIR
from retail_forecast.data.load import melt_to_long

STORE_ID = "CA_1"

plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
# Load raw files and filter CA_1
sales_raw = pd.read_csv(RAW_DATA_DIR / "sales_train_evaluation.csv")
calendar  = pd.read_csv(RAW_DATA_DIR / "calendar.csv")
prices    = pd.read_csv(RAW_DATA_DIR / "sell_prices.csv")

ca1_wide = sales_raw[sales_raw["store_id"] == STORE_ID].reset_index(drop=True)
prices_ca1 = prices[prices["store_id"] == STORE_ID]

print(f"CA_1 items : {len(ca1_wide):,}")
print(f"Categories : {sorted(ca1_wide['cat_id'].unique())}")
print(f"Departments: {sorted(ca1_wide['dept_id'].unique())}")
print(f"Day columns: {sum(c.startswith('d_') for c in ca1_wide.columns)}")

In [ ]:
# Melt to long format and join calendar
long = melt_to_long(ca1_wide, calendar_df=calendar)

# Join sell prices
long = long.merge(
    prices_ca1[["item_id", "wm_yr_wk", "sell_price"]],
    on=["item_id", "wm_yr_wk"],
    how="left",
)

print(f"Long shape    : {long.shape}")
print(f"Date range    : {long['date'].min().date()} => {long['date'].max().date()}")
print(f"Memory        : {long.memory_usage(deep=True).sum() / 1e9:.2f} GB")

---
## 1. Category × Department Breakdown

In [ ]:
# Item counts per category and department
item_counts = (
    ca1_wide.groupby(["cat_id", "dept_id"])
    .size()
    .reset_index(name="n_items")
)
print(item_counts.to_string(index=False))

In [ ]:
# Total sales per department
dept_total = (
    long.groupby(["cat_id", "dept_id"])["sales"]
    .sum()
    .reset_index()
    .sort_values("sales", ascending=False)
)

fig = px.bar(
    dept_total,
    x="dept_id",
    y="sales",
    color="cat_id",
    title="CA_1 - Total Sales by Department",
    labels={"sales": "Total units sold", "dept_id": "Department", "cat_id": "Category"},
)
fig.show()

---
## 2. Zero-Inflation Analysis

Zero-inflation is the core motivation for using LightGBM with **Tweedie objective** instead of standard L2 (Gaussian) loss.  
Gaussian regression minimises squared error and assumes a symmetric distribution - it will systematically over-predict on high-sparsity series.

In [ ]:
# Zero-inflation rate by category
zero_by_cat = (
    long.groupby("cat_id")["sales"]
    .apply(lambda s: (s == 0).mean() * 100)
    .reset_index()
    .rename(columns={"sales": "zero_pct"})
    .sort_values("zero_pct", ascending=False)
)

fig = px.bar(
    zero_by_cat,
    x="cat_id",
    y="zero_pct",
    color="cat_id",
    title="CA_1 - Zero-Sales Rate by Category (%)",
    labels={"zero_pct": "Zero-sales rate (%)", "cat_id": "Category"},
    text="zero_pct",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.show()

print(zero_by_cat.to_string(index=False))

In [ ]:
# Zero-inflation rate by department
zero_by_dept = (
    long.groupby(["cat_id", "dept_id"])["sales"]
    .apply(lambda s: (s == 0).mean() * 100)
    .reset_index()
    .rename(columns={"sales": "zero_pct"})
    .sort_values("zero_pct", ascending=False)
)

fig = px.bar(
    zero_by_dept,
    x="dept_id",
    y="zero_pct",
    color="cat_id",
    title="CA_1 - Zero-Sales Rate by Department (%)",
    labels={"zero_pct": "Zero-sales rate (%)", "dept_id": "Department", "cat_id": "Category"},
    text="zero_pct",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.show()

In [ ]:
# Per-item zero-inflation distribution - one histogram per category
zero_by_item = (
    long.groupby(["item_id", "cat_id"])["sales"]
    .apply(lambda s: (s == 0).mean() * 100)
    .reset_index()
    .rename(columns={"sales": "zero_pct"})
)

fig = px.histogram(
    zero_by_item,
    x="zero_pct",
    color="cat_id",
    facet_col="cat_id",
    nbins=40,
    title="CA_1 - Per-Item Zero-Sales Rate Distribution by Category",
    labels={"zero_pct": "Zero-sales rate per item (%)", "cat_id": "Category"},
)
fig.update_layout(showlegend=False)
fig.show()

print("Median zero-inflation per category:")
print(zero_by_item.groupby("cat_id")["zero_pct"].median().round(1).to_string())

---
## 3. Intermittent Demand Profiling

We pick one representative item from each category to visualise the **demand sparsity spectrum**:
- **HOBBIES** - ultra-sparse (many zero-sale days)
- **HOUSEHOLD** - medium sparsity
- **FOODS** - dense (few zeros)

This spectrum is why Tweedie dominates on HOBBIES/HOUSEHOLD but the advantage narrows on FOODS.
Three categories in CA_1 let us measure exactly *where* Tweedie wins.

In [ ]:
# Pick the item closest to the median zero-inflation rate for each category
def pick_median_item(cat: str) -> str:
    cat_items = zero_by_item[zero_by_item["cat_id"] == cat]
    median_val = cat_items["zero_pct"].median()
    idx = (cat_items["zero_pct"] - median_val).abs().idxmin()
    return cat_items.loc[idx, "item_id"]

sample_items = {cat: pick_median_item(cat) for cat in ["HOBBIES", "HOUSEHOLD", "FOODS"]}
print("Representative items (closest to median zero-inflation per category):")
for cat, item in sample_items.items():
    z = zero_by_item.set_index("item_id").loc[item, "zero_pct"]
    print(f"  {cat}: {item}  (zero-inflation: {z:.1f}%)")

In [ ]:
# Time series of representative items
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=[
        f"HOBBIES - {sample_items['HOBBIES']}",
        f"HOUSEHOLD - {sample_items['HOUSEHOLD']}",
        f"FOODS - {sample_items['FOODS']}",
    ],
    vertical_spacing=0.08,
)

colors = {"HOBBIES": "#EF553B", "HOUSEHOLD": "#00CC96", "FOODS": "#636EFA"}

for row, (cat, item_id) in enumerate(sample_items.items(), start=1):
    ts = long[long["item_id"] == item_id][["date", "sales"]].copy()
    fig.add_trace(
        go.Scatter(
            x=ts["date"], y=ts["sales"],
            mode="lines",
            line=dict(color=colors[cat], width=1),
            name=cat,
        ),
        row=row, col=1,
    )
    fig.update_yaxes(title_text="Units", row=row, col=1)

fig.update_layout(
    title="CA_1 - Intermittent Demand Spectrum (one item per category)",
    height=600,
    showlegend=False,
    xaxis3_title="Date",
)
fig.show()

In [ ]:
# Sales value distribution (log scale) - shows the zero-inflated, long-tail shape
fig = px.histogram(
    long[long["sales"] > 0],  # exclude zeros to see the positive-value distribution
    x="sales",
    color="cat_id",
    facet_col="cat_id",
    nbins=60,
    log_y=True,
    title="CA_1 - Distribution of Non-Zero Sales by Category (log y-axis)",
    labels={"sales": "Daily units sold (non-zero)", "cat_id": "Category"},
)
fig.update_layout(showlegend=False)
fig.show()

---
## 4. Price Dynamics

Sell prices change over time. Price features (absolute price, relative-to-year average, pct change) will be important for the model.

In [ ]:
# Price distribution by category
price_data = long.dropna(subset=["sell_price"])

fig = px.box(
    price_data,
    x="cat_id",
    y="sell_price",
    color="cat_id",
    title="CA_1 - Sell Price Distribution by Category",
    labels={"sell_price": "Sell price ($)", "cat_id": "Category"},
    points=False,
)
fig.show()

In [ ]:
# Median price over time by category
price_trend = (
    price_data
    .groupby(["date", "cat_id"])["sell_price"]
    .median()
    .reset_index()
)

fig = px.line(
    price_trend,
    x="date",
    y="sell_price",
    color="cat_id",
    title="CA_1 - Median Sell Price Over Time by Category",
    labels={"sell_price": "Median sell price ($)", "date": "Date", "cat_id": "Category"},
)
fig.show()

In [ ]:
# Price vs sales correlation - do lower prices drive higher volume?
# Sample to keep the scatter manageable
sample = price_data.sample(n=min(50_000, len(price_data)), random_state=42)

fig = px.scatter(
    sample,
    x="sell_price",
    y="sales",
    color="cat_id",
    facet_col="cat_id",
    opacity=0.15,
    trendline="lowess",
    title="CA_1 - Price vs Sales (sampled, with LOWESS trend)",
    labels={"sell_price": "Sell price ($)", "sales": "Units sold", "cat_id": "Category"},
)
fig.update_layout(showlegend=False)
fig.show()

---
## 5. SNAP & Event Effects (CA_1 Specific)

SNAP benefits are California-specific (`snap_CA`). They primarily boost FOODS sales.

In [ ]:
# SNAP effect on each category in CA_1
if "snap_CA" in long.columns:
    snap_effect = (
        long.groupby(["snap_CA", "cat_id"])["sales"]
        .mean()
        .reset_index()
    )
    snap_effect["day_type"] = snap_effect["snap_CA"].map({1: "SNAP day", 0: "Normal day"})

    fig = px.bar(
        snap_effect,
        x="cat_id",
        y="sales",
        color="day_type",
        barmode="group",
        title="CA_1 - SNAP Day Effect on Average Sales per Category",
        labels={"sales": "Avg. units sold", "cat_id": "Category", "day_type": "Day type"},
    )
    fig.show()

    snap_lift = (
        snap_effect.set_index(["cat_id", "day_type"])["sales"]
        .unstack()
        .assign(lift_pct=lambda df: (df["SNAP day"] / df["Normal day"] - 1) * 100)
    )
    print("SNAP lift % by category:")
    print(snap_lift[["lift_pct"]].round(1).to_string())

In [ ]:
# Top events by average sales impact - CA_1 only
event_rows = long[long["event_name_1"].notna()]
normal_avg = long[long["event_name_1"].isna()]["sales"].mean()

event_avg = (
    event_rows.groupby("event_name_1")["sales"]
    .mean()
    .reset_index()
    .rename(columns={"sales": "avg_sales"})
    .assign(lift_vs_normal=lambda df: (df["avg_sales"] / normal_avg - 1) * 100)
    .sort_values("lift_vs_normal", ascending=False)
    .head(15)
)

fig = px.bar(
    event_avg,
    x="event_name_1",
    y="lift_vs_normal",
    title="CA_1 - Top 15 Events: Sales Lift vs Normal Days (%)",
    labels={"lift_vs_normal": "Sales lift vs normal day (%)", "event_name_1": "Event"},
    color="lift_vs_normal",
    color_continuous_scale="RdYlGn",
)
fig.update_layout(xaxis_tickangle=-40, coloraxis_showscale=False)
fig.show()

---
## 6. Train / Validation / Test Split

Chronological split - no shuffling, no leakage.
- **Train:** first ~80% of days
- **Validation:** next ~10% (used for hyperparameter selection)
- **Test (holdout):** last ~10% (never seen during training)

In [ ]:
dates = long["date"].sort_values().unique()
n = len(dates)
train_end = dates[int(n * 0.80)]
val_end   = dates[int(n * 0.90)]
test_end  = dates[-1]

print(f"Total days : {n}")
print(f"Train      : up to {pd.Timestamp(train_end).date()}  ({int(n * 0.80)} days)")
print(f"Validation : {pd.Timestamp(train_end).date()} => {pd.Timestamp(val_end).date()}  ({int(n * 0.10)} days)")
print(f"Test       : {pd.Timestamp(val_end).date()} => {pd.Timestamp(test_end).date()}  ({n - int(n * 0.90)} days)")

# Aggregate daily sales for the CA_1 subset to plot the split
daily = long.groupby("date")["sales"].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 4))

def _plot_segment(mask, color, label):
    seg = daily[mask]
    ax.fill_between(seg["date"], seg["sales"], alpha=0.3, color=color, label=label)
    ax.plot(seg["date"], seg["sales"], color=color, linewidth=0.8)

_plot_segment(daily["date"] <= train_end, "#636EFA", "Train (80%)")
_plot_segment((daily["date"] > train_end) & (daily["date"] <= val_end), "#FFA15A", "Validation (10%)")
_plot_segment(daily["date"] > val_end, "#EF553B", "Test - holdout (10%)")

ax.axvline(train_end, color="#FFA15A", linestyle="--", linewidth=1.2)
ax.axvline(val_end,   color="#EF553B", linestyle="--", linewidth=1.2)
ax.set_title("CA_1 - Total Daily Sales with Train / Validation / Test Split", fontsize=13)
ax.set_xlabel("Date")
ax.set_ylabel("Total units sold")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

---
## 7. Seasonal Pattern (CA_1)

Weekly and monthly cycles directly motivate the lag and calendar features.

In [ ]:
# Day-of-week pattern by category
if "weekday" in long.columns:
    weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    dow = (
        long.groupby(["weekday", "cat_id"])["sales"]
        .mean()
        .reset_index()
    )
    dow["weekday"] = pd.Categorical(dow["weekday"], categories=weekday_order, ordered=True)
    dow = dow.sort_values("weekday")

    fig = px.line(
        dow,
        x="weekday",
        y="sales",
        color="cat_id",
        markers=True,
        title="CA_1 - Average Daily Sales by Day of Week and Category",
        labels={"sales": "Avg. units sold", "weekday": "Day of week", "cat_id": "Category"},
    )
    fig.show()

In [ ]:
# Month-of-year pattern
if "month" in long.columns:
    monthly = (
        long.groupby(["month", "cat_id"])["sales"]
        .mean()
        .reset_index()
    )

    fig = px.line(
        monthly,
        x="month",
        y="sales",
        color="cat_id",
        markers=True,
        title="CA_1 - Average Daily Sales by Month and Category",
        labels={"sales": "Avg. units sold", "month": "Month", "cat_id": "Category"},
    )
    fig.update_xaxes(tickvals=list(range(1, 13)),
                     ticktext=["Jan","Feb","Mar","Apr","May","Jun",
                               "Jul","Aug","Sep","Oct","Nov","Dec"])
    fig.show()

---
## 8. Conclusions & Feature Engineering Implications

### Zero-inflation
- HOBBIES: **~72%**
- HOUSEHOLD: **~68%**
- FOODS: **~57%**

### Tweedie motivation
Gaussian (L2) regression will over-predict zero-demand periods because it minimises squared residuals symmetrically.  
Tweedie with `variance_power ~ 1.1–1.5` assigns a compound Poisson–Gamma distribution that naturally handles the spike at zero + the long right tail.  
The intermittency spectrum across three categories lets us quantify *where* Tweedie wins (=> Notebook 04).

### Feature engineering priorities
| Feature | Motivation |
|---|---|
| `lag_7`, `lag_14`, `lag_28` | Weekly / bi-weekly demand cycles |
| `rolling_mean_7`, `rolling_mean_28` | Local trend, smoothing |
| `rolling_std_28` | Demand volatility (important for Tweedie power choice) |
| `dow`, `month`, `is_weekend` | Day-of-week and monthly seasonality |
| `snap_CA` | CA SNAP benefit days boost FOODS |
| `event_type_1` | Event-driven spikes |
| `sell_price`, `price_rel_year` | Price elasticity effects |

---
## 9. Anomaly Detection - MSTL + MAD (per store–item pair)

**MSTL** (Multiple Seasonal-Trend decomposition via Loess) decomposes each item's time series into trend, weekly seasonal component, and residual.  
**MAD** (Median Absolute Deviation) then flags residuals that deviate too far from the median - it is more robust than std for zero-inflated, heavy-tailed retail data.

Modified Z-score formula:  
`modified_z = 0.6745 × (residual − median(residual)) / MAD`  
A day is flagged as anomalous when `|modified_z| > 3.5`.

> ⏱ **Runtime note:** MSTL is fit independently for all 3,049 items (~3–5 min depending on hardware).

In [ ]:
import warnings
from statsmodels.tsa.seasonal import MSTL

MAD_THRESHOLD = 3.5

def _mstl_mad(series: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Fit MSTL (weekly seasonality) on a single item's sales series.
    Return (residuals, is_anomaly) using MAD-based modified Z-score.
    """
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        res = MSTL(series, periods=[7]).fit()

    resid = np.asarray(res.resid, dtype=float)
    median = np.median(resid)
    mad = np.median(np.abs(resid - median))

    if mad == 0:
        return resid, np.zeros(len(resid), dtype=bool)

    modified_z = 0.6745 * (resid - median) / mad
    return resid, np.abs(modified_z) > MAD_THRESHOLD


# Build a lookup: item_id => (cat_id, sorted date array)
item_meta = (
    long[["item_id", "cat_id"]]
    .drop_duplicates("item_id")
    .set_index("item_id")["cat_id"]
    .to_dict()
)

# Run MSTL + MAD for every item
records = []         # summary: one row per item
flag_rows = []       # full flags: one row per (item, date)

items_sorted = long.sort_values(["item_id", "date"])

for item_id, grp in items_sorted.groupby("item_id", sort=False):
    dates  = grp["date"].values
    sales  = grp["sales"].values.astype(float)

    resid, is_anom = _mstl_mad(sales)

    records.append({
        "item_id":     item_id,
        "cat_id":      item_meta[item_id],
        "n_anomalies": int(is_anom.sum()),
        "pct_anomaly": float(is_anom.mean() * 100),
    })
    flag_rows.append(
        pd.DataFrame({"item_id": item_id, "date": dates,
                      "sales": sales, "residual": resid, "is_anomaly": is_anom})
    )

anomaly_summary = pd.DataFrame(records)
anomaly_flags   = pd.concat(flag_rows, ignore_index=True)

print(f"Items processed : {len(anomaly_summary):,}")
print(f"Total anomalies : {anomaly_summary['n_anomalies'].sum():,}")
print(f"Avg per item    : {anomaly_summary['n_anomalies'].mean():.1f} days")
print(f"\nAnomaly count by category:")
print(anomaly_summary.groupby("cat_id")["n_anomalies"].agg(["sum", "mean"]).round(1))